# ggplot2 style

For those who know ggplot2's faceting:

| ggplot2                                  | anytrellis                                  |
|------------------------------------------|---------------------------------------------|
| `facet_wrap(~ a, ncol = 3)`              | `Trellis(df, wrap="a", cols=3)`             |
| `facet_grid(a ~ b)`                      | `Trellis(df, row="a", col="b")`             |
| `scales = "free_y"` / `"free"`           | `share_y=False` / `share_x=False, share_y=False` |
| `geom_point()`, `geom_line()`, `geom_col()` | `.points()`, `.line()`, `.bars()`        |
| `geom_hline()`, `geom_vline()`           | `.rule(y=...)`, `.rule(x=...)`              |
| `position = "dodge"`                     | `.bars(position="dodge")`                   |
| `gghighlight()`                          | a layer with `facet="exclude"`              |
| `fct_reorder(f, x, median)`              | `.order("f", by="median", of="x")`          |

**Using a plot:** click it, then scroll to zoom along x (Shift+scroll: y), drag to pan, and
press `r` to reset. Panels that share a scale zoom and pan together. Hover a mark for its
values. Double-click a panel to see it alone (Page Up and Page Down: the next ones;
Backspace: all again), and press `f` (or the toolbar's button) for full screen.

In [ ]:
import polars as pl

from anytrellis import Trellis
from anytrellis.datasets import load

## `facet_wrap`

```r
ggplot(penguins, aes(flipper_length_mm, body_mass_g, colour = sex)) +
  geom_point() + facet_wrap(~ species)
```

Rows without a sex get a `(missing)` colour of their own.

In [ ]:
penguins = load("penguins")

wrapped = (
    Trellis(penguins, wrap="species", title="Palmer penguins")
    .points(x="flipper_length_mm", y="body_mass_g", color="sex", size=5, tooltip=["island"])
)
wrapped

## `facet_grid`

```r
ggplot(drop_na(penguins, sex), aes(bill_length_mm, bill_depth_mm)) +
  geom_point() + facet_grid(sex ~ species)
```

In [ ]:
known = penguins.drop_nulls("sex")

crossed = (
    Trellis(known, row="sex", col="species", panel_height=160)
    .points(x="bill_length_mm", y="bill_depth_mm", color="island", size=5)
)
crossed

## `gghighlight`: each group against the rest

```r
ggplot(gapminder, aes(year, life_expect, group = country, colour = region)) +
  geom_line() + gghighlight() + facet_wrap(~ region)
```

The grey layer (`facet="exclude"`) draws every other region's countries behind each
panel.

In [ ]:
gapminder = load("gapminder")

highlighted = (
    Trellis(gapminder, wrap="region", cols=3, title="Life expectancy, by region")
    .line(x="year", y="life_expect", group="country", facet="exclude", width=1, name="others")
    .line(x="year", y="life_expect", group="country", color="region", tooltip=["country"])
)
highlighted

## `scales = "free_y"`

Population differs by orders of magnitude between regions: free y scales show each
region's trend (and the tick labels say how far apart they are).

In [ ]:
population = (
    Trellis(gapminder, wrap="region", cols=3, share_y=False, title="Population")
    .line(x="year", y="pop", group="country", width=1, tooltip=["country"])
)
population

## `geom_col` with stacked categories

Days of each kind of weather in Seattle per month, a panel per year. The months and the
kinds of weather are put in order with `order()`, and `palette=` gives the kinds their
colours (in that order).

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
kinds = ["sun", "fog", "drizzle", "rain", "snow"]

monthly = (
    load("seattle_weather")
    .group_by("weather", year=pl.col("date").dt.year(), month=pl.col("date").dt.strftime("%b"))
    .agg(days=pl.len())
)

stacked = (
    Trellis(monthly, wrap="year", cols=2, title="Seattle weather",
            palette=["#e7ba52", "#c7c7c7", "#aec7e8", "#1f77b4", "#9467bd"])
    .order("month", months)
    .order("weather", kinds)
    .bars(x="month", y="days", color="weather")
)
stacked

## `position = "dodge"`

```r
penguins |> group_by(island, species, sex) |> summarise(mass = mean(body_mass_g)) |>
  ggplot(aes(species, mass, fill = sex)) + geom_col(position = "dodge") + facet_grid(~ island)
```

In [ ]:
mass = known.group_by("island", "species", "sex").agg(mass=pl.col("body_mass_g").mean())

dodged = (
    Trellis(mass, col="island", panel_height=220, title="Mean body mass", y_title="grams")
    .bars(x="species", y="mass", color="sex", position="dodge")
)
dodged